# ResNet18 za oxford_flowers102


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU PRESENT")

In [ ]:
# podaci
!mkdir -p data && cd data && \
  curl -sL -O https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz && \
  curl -sL -O https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat && \
  tar -xzf 102flowers.tgz
!ls data/jpg | wc -l

In [ ]:
import os, json, numpy as np, pandas as pd, scipy.io
import torch, torch.nn as nn
from pathlib import Path
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("assets", exist_ok=True)
print(device)

In [ ]:
DROP_03540 = True

IMG_DIR = Path('data/jpg')
labels_raw = scipy.io.loadmat('data/imagelabels.mat')['labels'].flatten()
image_paths = sorted(IMG_DIR.glob('*.jpg'), key=lambda p: int(p.stem.split('_')[1]))
df = pd.DataFrame({'path': image_paths, 'label': labels_raw})

corrupt = []
for p in image_paths:
    try:
        with Image.open(p) as img:
            _ = img.size, img.mode
        corrupt.append(False)
    except (UnidentifiedImageError, OSError):
        corrupt.append(True)
df['corrupt'] = corrupt
df = df[~df['corrupt']].drop(columns=['corrupt'])

if DROP_03540:
    df = df[df['path'].apply(lambda p: p.name) != 'image_03540.jpg']

X = df['path'].values
y = df['label'].values
idx = np.arange(len(df))

X_train_val, X_test, y_train_val, y_test, itr, ite = train_test_split(X, y, idx, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val, itr, iva = train_test_split(X_train_val, y_train_val, itr, test_size=0.15 / 0.85, stratify=y_train_val, random_state=42)

num_classes = len(np.unique(df['label'].values))
label_to_idx = {label: i for i, label in enumerate(sorted(np.unique(df['label'].values)))}
idx_to_label = {v: k for k, v in label_to_idx.items()}
target_names = [str(idx_to_label[i]) for i in range(num_classes)]

print(f"slika {len(df)} | train {len(X_train)} val {len(X_val)} test {len(X_test)} | klasa {num_classes}")

In [ ]:
# smanjenje slika na 256 piksela zbog brzine
SMALL_DIR = Path('data/jpg256')
if not SMALL_DIR.exists():
    SMALL_DIR.mkdir()
    for p in image_paths:
        with Image.open(p) as im:
            im.convert('RGB').resize((256, 256), Image.BILINEAR).save(SMALL_DIR / p.name, quality=92)
    print("shrunk:", len(list(SMALL_DIR.glob('*.jpg'))))

def small(paths):
    return np.array([SMALL_DIR / Path(p).name for p in paths])

X_train_s, X_val_s, X_test_s = small(X_train), small(X_val), small(X_test)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
RESNET_IMG_SIZE = 224
BATCH_SIZE = 32

class FlowerDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = label_to_idx[self.labels[idx]]
        return img, label

rn_train_transform = transforms.Compose([
    transforms.Resize((RESNET_IMG_SIZE, RESNET_IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
rn_eval_transform = transforms.Compose([
    transforms.Resize((RESNET_IMG_SIZE, RESNET_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

rn_train_loader = DataLoader(FlowerDataset(X_train_s, y_train, transform=rn_train_transform),
                             batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
rn_val_loader = DataLoader(FlowerDataset(X_val_s, y_val, transform=rn_eval_transform),
                           batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
rn_test_loader = DataLoader(FlowerDataset(X_test_s, y_test, transform=rn_eval_transform),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(rn_train_loader)}, Val batches: {len(rn_val_loader)}, Test batches: {len(rn_test_loader)}")

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, step_counter=None, metrics=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if is_train:
                optimizer.zero_grad()

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            batch_size = imgs.size(0)
            batch_loss = loss.item()
            preds = outputs.argmax(dim=1)
            batch_correct = (preds == labels).sum().item()
            batch_acc = batch_correct / batch_size

            total_loss += batch_loss * batch_size
            total_correct += batch_correct
            total_samples += batch_size

            if is_train and metrics is not None and step_counter is not None:
                step_counter[0] += 1
                metrics["train_loss"].append(batch_loss)
                metrics["train_acc"].append(batch_acc)
                metrics["train_steps"].append(step_counter[0])

    return total_loss / total_samples, total_correct / total_samples


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=100, patience=7, save_path="assets/best_cnn_model.pt"):
    best_val_acc = 0.0
    patience_counter = 0

    metrics = {"train_loss": [], "train_acc": [], "train_steps": [], "val_loss": [], "val_acc": [], "val_steps": []}
    global_step = [0]
    for epoch in range(1, num_epochs + 1):

        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, step_counter=global_step, metrics=metrics)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

        scheduler.step(val_acc)

        metrics["val_loss"].append(val_loss)
        metrics["val_acc"].append(val_acc)
        metrics["val_steps"].append(global_step[0])

        print(f"Epoch {epoch}/{num_epochs}: "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    return best_val_acc, metrics

In [ ]:
def freeze_bn(model):
    def train(mode=True):
        nn.Module.train(model, mode)
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d) and not m.weight.requires_grad:
                m.eval()
        return model
    model.train = train
    return model

In [ ]:
rn = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in rn.parameters():
    param.requires_grad = False
rn.fc = nn.Linear(rn.fc.in_features, num_classes)
rn = freeze_bn(rn.to(device))

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(rn.fc.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

RESNET_HEAD_PATH = "assets/resnet18_head.pt"
best_val_acc_head, metrics_head = train_model(rn, rn_train_loader, rn_val_loader, criterion,
                                              optimizer, scheduler, num_epochs=8, patience=3,
                                              save_path=RESNET_HEAD_PATH)
print(f"Best validation accuracy, frozen backbone: {best_val_acc_head}")

In [ ]:
for name, param in rn.named_parameters():
    param.requires_grad = name.startswith(("layer3", "layer4", "fc"))

optimizer = torch.optim.Adam([p for p in rn.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

RESNET_PATH = "assets/resnet18_finetuned.pt"
RESNET_METRICS_PATH = "assets/resnet18_metrics.json"

best_val_acc_rn, metrics_rn = train_model(rn, rn_train_loader, rn_val_loader, criterion,
                                          optimizer, scheduler, num_epochs=15, patience=5,
                                          save_path=RESNET_PATH)
print(f"Best validation accuracy, fine-tuned: {best_val_acc_rn}")

with open(RESNET_METRICS_PATH, "w") as f:
    json.dump(metrics_rn, f, indent=2)

In [ ]:
rn.load_state_dict(torch.load(RESNET_PATH, map_location=device, weights_only=True))
rn.eval()

all_preds_rn, all_labels_rn = [], []
with torch.no_grad():
    for imgs, labels in rn_test_loader:
        outputs = rn(imgs.to(device))
        all_preds_rn.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels_rn.extend(labels.numpy())

all_preds_rn = np.array(all_preds_rn)
all_labels_rn = np.array(all_labels_rn)

test_acc_rn = (all_preds_rn == all_labels_rn).mean()
print(f"Test accuracy for ResNet18: {test_acc_rn:.4f}")
print(f"Best validation accuracy, frozen backbone: {best_val_acc_head:.4f}")
print(f"Best validation accuracy, fine-tuned:      {best_val_acc_rn:.4f}")

cm_rn = confusion_matrix(all_labels_rn, all_preds_rn)
per_class = cm_rn.diagonal() / cm_rn.sum(axis=1)
print(f"Macro accuracy per class: {per_class.mean():.4f}")
print(f"Classes below 0.5: {(per_class < 0.5).sum()}")
print()
print(classification_report(all_labels_rn, all_preds_rn, target_names=target_names, digits=4, zero_division=0))

In [ ]:
!cd assets && zip -q resnet18_za_assets.zip resnet18_head.pt resnet18_finetuned.pt resnet18_metrics.json && ls -la resnet18_za_assets.zip
from google.colab import files #download sa google collab sajta
files.download('assets/resnet18_za_assets.zip')